# Orbit and Coverage Diagnostics in v0.13

Theme: explain the new public `pdelie.invariants` diagnostics visually and precisely.

This notebook is the v0.13 feature notebook. It deliberately does **not** construct augmented datasets. It reports what a translation-window workflow would cover and whether the uniform translation action behaves consistently.


> Run from the repository root after `python -m pip install -e .[test]`.  
> These notebooks are tutorials, not package contracts.  They use public runtime APIs unless a cell explicitly says otherwise.


In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np

from pdelie.data import generate_heat_1d_field_batch, generate_kdv_1d_field_batch
from pdelie.examples import run_orbit_coverage_diagnostics_example
from pdelie.invariants import compute_periodic_window_coverage, diagnose_uniform_translation_consistency
from pdelie.residuals import HeatResidualEvaluator, KdVResidualEvaluator


## 1. The shift convention in one picture

Positive shift means: translate the field by `shift`, then observe a fixed window. In original coordinates, the covered preimage moves by `-shift`.


In [ ]:
L = 2.0 * np.pi
x = np.linspace(0.0, L, 64, endpoint=False)
window = {'start': 0.0, 'width': L / 8.0}
shift = L / 8.0
coverage = compute_periodic_window_coverage(x=x, windows=[window], shifts=[shift], domain_length=L)

fig, ax = plt.subplots(figsize=(9, 2.8))
ax.scatter(x, coverage['coverage_counts'], c=coverage['coverage_counts'], cmap='viridis', s=55)
ax.set_title('Positive shift covers the preimage [window_start - shift, window_start + width - shift)')
ax.set_xlabel('original x grid')
ax.set_ylabel('covered?')
ax.set_yticks([0, 1])
ax.grid(alpha=0.25)
plt.show()

{
    'coverage_convention': coverage['coverage_convention'],
    'shift_convention': coverage['shift_convention'],
    'coverage_counts': coverage['coverage_counts'],
}


## 2. Coverage is grid-point coverage

The report counts sampled grid points, not continuous interval measure. Duplicate shifts and repeated windows increase counts, not the number of unique covered grid points.


In [ ]:
half = compute_periodic_window_coverage(
    x=x,
    windows=[{'start': 0.0, 'width': L / 8.0}],
    shifts=[0.0, L / 4.0, L / 2.0, 3.0 * L / 4.0],
    domain_length=L,
)
full = compute_periodic_window_coverage(
    x=x,
    windows=[{'start': 0.0, 'width': L / 4.0}],
    shifts=[0.0, L / 4.0, L / 2.0, 3.0 * L / 4.0],
    domain_length=L,
)
duplicate = compute_periodic_window_coverage(
    x=x,
    windows=[{'start': 0.0, 'width': L / 8.0}, {'start': 0.0, 'width': L / 8.0}],
    shifts=[0.0, 0.0],
    domain_length=L,
)

fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
for ax, title, summary in [
    (axes[0], 'half coverage', half),
    (axes[1], 'full coverage', full),
    (axes[2], 'duplicate windows and shifts', duplicate),
]:
    ax.step(x, summary['coverage_counts'], where='post')
    ax.set_title(f"{title}: fraction={summary['coverage_fraction']}")
    ax.set_ylabel('count')
    ax.grid(alpha=0.25)
axes[-1].set_xlabel('x')
plt.tight_layout()
plt.show()


## 3. Translation consistency is a report, not a transformed dataset

The consistency helper internally creates transformed fields, but the public output is a JSON-compatible report only.


In [ ]:
heat = generate_heat_1d_field_batch(batch_size=2, num_times=17, num_points=64, seed=1301)
kdv = generate_kdv_1d_field_batch(batch_size=2, num_times=17, num_points=64, seed=1302)
shifts = [0.0, L / 64.0, L / 8.0, -L / 8.0, L]

heat_consistency = diagnose_uniform_translation_consistency(
    heat,
    shifts=shifts,
    residual_evaluator=HeatResidualEvaluator(),
)
kdv_consistency = diagnose_uniform_translation_consistency(
    kdv,
    shifts=shifts,
    residual_evaluator=KdVResidualEvaluator(),
)

compact = {}
for name, summary in {'heat': heat_consistency, 'kdv': kdv_consistency}.items():
    compact[name] = [
        {
            'shift': report['shift'],
            'inverse_error': report['inverse_relative_l2_error'],
            'period_wrap_error': report['period_wrap_relative_l2_error'],
            'residual_relative_delta': report['residual_relative_rms_delta'],
            'provenance': report['provenance_construction_method'],
        }
        for report in summary['shift_reports']
    ]
compact


## 4. Visualize residual stability under shifts

The pass rule is intentionally robust near small baseline residuals: absolute delta may pass even when relative delta is noisy.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
for name, summary in {'heat': heat_consistency, 'kdv': kdv_consistency}.items():
    shift_values = [report['shift'] for report in summary['shift_reports']]
    absolute_delta = [report['residual_absolute_rms_delta'] for report in summary['shift_reports']]
    ax.plot(shift_values, absolute_delta, marker='o', label=name)
ax.axhline(1e-8, color='black', linestyle='--', linewidth=1, label='absolute tolerance')
ax.set_yscale('log')
ax.set_title('Residual RMS absolute delta under uniform translations')
ax.set_xlabel('shift')
ax.set_ylabel('absolute RMS delta')
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## 5. The packaged example is a smoke summary

The example combines the same two diagnostics. It is useful for smoke testing and teaching, not a canonical artifact schema.


In [ ]:
example = run_orbit_coverage_diagnostics_example()
print(json.dumps({
    'summary_type': example['summary_type'],
    'coverage_fractions': [case['coverage_fraction'] for case in example['coverage_cases']],
    'transform_fixtures': example['extra_metrics']['transform_fixtures'],
}, indent=2))


## Takeaway

The v0.13 diagnostics give you two reusable questions:

- coverage: *which grid points are observed by this translated-window design?*
- consistency: *does the finite translation action preserve structure, inverse behavior, period wrapping, provenance, and residual scale?*

They intentionally stop there. Dataset augmentation and experiment policy remain downstream or future scope.
